# Flash desktop gratis (noVNC)
Ejecutá las celdas en orden. Al final tendrás una URL pública trycloudflare.com para abrir el escritorio.

In [ ]:
!apt-get -y update
!apt-get -y install xfce4 xfce4-terminal novnc websockify x11vnc xvfb wget curl
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

In [ ]:
%%bash
set -e
mkdir -p ~/apps/flashbrowser ~/Desktop
cd ~/apps/flashbrowser
# Descarga paquete Linux y extrae binario
ZIP_URLS=(
  "https://github.com/radubirsan/FlashBrowser/releases/download/v0.7/FlashBrowser-linux-x64.zip"
  "https://github.com/radubirsan/FlashBrowser/releases/download/v0.01/FlashBrowser-linux-x64.zip"
)
ok=0
for Z in "${ZIP_URLS[@]}"; do
  echo "Trying $Z"
  if curl -L --fail -o fb.zip "$Z"; then
    ok=1; break
  fi
done
if [ "$ok" != "1" ]; then echo "No se pudo descargar FlashBrowser para Linux"; exit 1; fi
apt-get -y install unzip >/dev/null 2>&1 || true
unzip -o fb.zip -d ./unz >/dev/null 2>&1 || true
# Busca el ejecutable dentro del zip
TARGET=$(find ./unz . -type f \( -name 'FlashBrowser' -o -name 'FlashBrowser*' -o -name '*AppImage' -o -name 'flashbrowser*' \) | head -n1)
if [ -z "$TARGET" ]; then echo "No se encontró binario FlashBrowser dentro del zip"; ls -R ./unz | head -n 200; exit 1; fi
cp "$TARGET" ./FlashBrowser
chmod +x FlashBrowser
cat > ~/Desktop/FlashBrowser.desktop <<'EOF'
[Desktop Entry]
Type=Application
Name=FlashBrowser
Exec=/root/apps/flashbrowser/FlashBrowser
Terminal=false
Categories=Game;
EOF
chmod +x ~/Desktop/FlashBrowser.desktop

In [ ]:
%%bash
export DISPLAY=:1
pkill -f x11vnc || true
pkill -f Xvfb || true
pkill -f websockify || true
Xvfb :1 -screen 0 1366x768x24 &
sleep 2
x11vnc -display :1 -nopw -forever -shared -rfbport 5900 &
websockify --web=/usr/share/novnc/ 6080 localhost:5900 &
startxfce4 &
sleep 5
if [ -x /root/apps/flashbrowser/FlashBrowser ]; then DISPLAY=:1 /root/apps/flashbrowser/FlashBrowser & fi
nohup cloudflared tunnel --no-autoupdate --url http://localhost:6080 > /content/cloudflared.log 2>&1 &
for i in $(seq 1 30); do
  URL=$(grep -o 'https://[a-zA-Z0-9.-]*trycloudflare.com' /content/cloudflared.log | tail -n1 || true)
  [ -n "$URL" ] && break
  sleep 2
done
echo "PUBLIC_URL=$URL"